## Imports

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


from src.model import build_hdea_model, nll_loss

import gc
import numpy as np
from src.utils.labeling import load_label_dict
from src.utils.preprocessing import prepare_dataset, normalize_features
from sklearn.model_selection import train_test_split

## Data Loading and Preprocessing

In [2]:
print("Loading and preprocessing data...")

# Load and preprocess data
label_dict = load_label_dict("../data/stream_labels.csv")
print(f"Loaded {len(label_dict)} labels")

# Prepare dataset
print("Preparing dataset (this may take a while)...")
X_raw, y = prepare_dataset("../data/SMARTPDM_dataset.csv", label_dict)
print(f"Dataset shape: X={X_raw.shape}, y={y.shape}")

# ✅ Scale RUL targets to range [0, 1]
max_rul = np.max(y)
print(f"Max RUL (for scaling): {max_rul}")
y_scaled = y / max_rul

# Check memory usage and use smaller sample if needed
total_size_gb = X_raw.nbytes / (1024**3)
print(f"Dataset size: {total_size_gb:.2f} GB")

if total_size_gb > 2:  # If dataset is larger than 2GB, sample it
    print("Dataset is large, sampling 30% for training...")
    sample_size = len(X_raw) // 3  # Use 1/3 of data for faster training
    indices = np.random.choice(len(X_raw), sample_size, replace=False)
    X_raw = X_raw[indices]
    y_scaled = y_scaled[indices]
    print(f"Sampled dataset shape: X={X_raw.shape}, y={y_scaled.shape}")

print("Normalizing features...")
X_norm, scaler = normalize_features(X_raw)

# Clear raw data to free memory
del X_raw
gc.collect()

print("Splitting dataset...")
# Split into train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y_scaled, test_size=0.3, stratify=y_scaled, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train set: X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")
print(f"RUL value ranges - Train: {y_train.min():.2f}–{y_train.max():.2f}, Val: {y_val.min():.2f}–{y_val.max():.2f}, Test: {y_test.min():.2f}–{y_test.max():.2f}")

# Clean up
del X_norm, X_temp, y_temp
gc.collect()

print("Data preprocessing completed successfully!")

Loading and preprocessing data...
Loaded 93 labels
Preparing dataset (this may take a while)...
Dataset shape: X=(423423, 2048, 2), y=(423423,)
Max RUL (for scaling): 1
Dataset size: 12.92 GB
Dataset is large, sampling 30% for training...
Dataset shape: X=(423423, 2048, 2), y=(423423,)
Max RUL (for scaling): 1
Dataset size: 12.92 GB
Dataset is large, sampling 30% for training...
Sampled dataset shape: X=(141141, 2048, 2), y=(141141,)
Normalizing features...
Sampled dataset shape: X=(141141, 2048, 2), y=(141141,)
Normalizing features...
Splitting dataset...
Splitting dataset...
Train set: X=(98798, 2048, 2), y=(98798,)
Validation set: X=(21171, 2048, 2), y=(21171,)
Test set: X=(21172, 2048, 2), y=(21172,)
RUL value ranges - Train: 0.00–1.00, Val: 0.00–1.00, Test: 0.00–1.00
Data preprocessing completed successfully!
Train set: X=(98798, 2048, 2), y=(98798,)
Validation set: X=(21171, 2048, 2), y=(21171,)
Test set: X=(21172, 2048, 2), y=(21172,)
RUL value ranges - Train: 0.00–1.00, Val: 0.

## Build Model

In [3]:
input_shape = X_train.shape[1:]  # (2048, 2)
model = build_hdea_model(input_shape)

2025-08-05 09:37:56.045490: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-08-05 09:37:56.045988: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-08-05 09:37:56.046841: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
I0000 00:00:1754375876.047447 44374108 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1754375876.047920 44374108 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
print("Model Summary:")
model.summary()

print("\nModel input shape:", model.input_shape)
print("Model output shapes:")
for i, output in enumerate(model.outputs):
    print(f"  Output {i} ({output.name}): {output.shape}")

Model Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 2048, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 2044, 32)  │        352 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 1022, 32)  │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 1020, 64)  │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ conv1d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rul_mean (Dense)    │ (None, 1)         │         65 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rul_log_var (Dense) │ (None, 1)         │         65 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,850 (42.38 KB)

 Trainable params: 10,850 (42.38 KB)

 Non-trainable params: 0 (0.00 B)


Model input shape: (None, 2048, 2)
Model output shapes:
  Output 0 (keras_tensor_6): (None, 1)
  Output 1 (keras_tensor_7): (None, 1)


## Custom Training Loop with Combined Loss

In [5]:
import tensorflow as tf

# Custom loss function that works with TensorFlow's graph execution
@tf.function
def combined_loss(y_true, y_pred):
    """
    Combined loss for multi-output model
    y_pred should be a list [mean, log_var]
    """
    return nll_loss(y_true, y_pred[0], y_pred[1])

# Alternative approach: Use separate losses for each output
def rul_mean_loss(y_true, y_pred_mean):
    return tf.reduce_mean(tf.square(y_true - y_pred_mean))

def rul_var_loss(y_true, y_pred_log_var):
    # Regularization loss for log variance
    return tf.reduce_mean(y_pred_log_var)

# Compile with separate losses
model.compile(
    loss={
        'rul_mean': rul_mean_loss,
        'rul_log_var': rul_var_loss
    },
    loss_weights={
        'rul_mean': 1.0,
        'rul_log_var': 0.1  # Smaller weight for variance regularization
    },
    optimizer='adam',
    metrics={
        'rul_mean': ['mse', 'mae'],
        'rul_log_var': ['mse']
    }
)

## Fit the Model

In [6]:
history = model.fit(
    x=X_train,
    y={
        'rul_mean': y_train,
        'rul_log_var': y_train  # Using same labels for both outputs
    },
    validation_data=(
        X_val, 
        {
            'rul_mean': y_val,
            'rul_log_var': y_val
        }
    ),
    epochs=10,  # Reduced epochs for faster training
    batch_size=32,  # Reduced batch size to avoid memory issues
    verbose=1
)

Epoch 1/10


2025-08-05 09:37:58.926559: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


3088/3088 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: -113318.5625 - rul_log_var_loss: -1173576.3750 - rul_log_var_mse: 7409322426368.0000 - rul_mean_loss: 4038.9319 - rul_mean_mae: 18.7953 - rul_mean_mse: 4038.9341 - val_loss: -1512583.2500 - val_rul_log_var_loss: -15126941.0000 - val_rul_log_var_mse: 265090867658752.0000 - val_rul_mean_loss: 7.9679 - val_rul_mean_mae: 2.4730 - val_rul_mean_mse: 7.9652
Epoch 2/10
3088/3088 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: -113318.5625 - rul_log_var_loss: -1173576.3750 - rul_log_var_mse: 7409322426368.0000 - rul_mean_loss: 4038.9319 - rul_mean_mae: 18.7953 - rul_mean_mse: 4038.9341 - val_loss: -1512583.2500 - val_rul_log_var_loss: -15126941.0000 - val_rul_log_var_mse: 265090867658752.0000 - val_rul_mean_loss: 7.9679 - val_rul_mean_mae: 2.4730 - val_rul_mean_mse: 7.9652
Epoch 2/10
3088/3088 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - loss: -2738345.2500 - rul_log_var_loss: -28229000.0000 - rul_log_var_mse: 1144659518685184.0000 - rul_mean_loss: 84553.